# IBM HR Analytics - Analise Exploratoria de Dados (EDA)

**Projeto:** Cegid Academy - Data Analytics
**Dataset:** IBM HR Analytics Attrition Dataset ([Kaggle](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset))
**Objetivo:** Analise exploratoria completa dos dados de RH para apoiar a reestruturacao organizacional

---

### Estrutura do Notebook

| Parte | Conteudo |
|:---:|:---|
| 0 | Setup do Ambiente e Carregamento dos Dados |
| 1 | Limpeza Inicial (remocao de colunas constantes) |
| 2 | Estatisticas Descritivas |
| 3 | Distribuicoes - Histogramas e Boxplots |
| 4 | Categoricas - Bar Plots |
| 5 | Correlacoes |
| 6 | Analise de Attrition (Turnover) |
| 7 | Comparacao Antes vs Depois (Ping Pong Survey) |

---

### Contexto do Projeto

O novo Gerente de RH precisa de conhecer a empresa de raiz - o anterior gerente faleceu e levou consigo todo o conhecimento que mantinha "na ponta da lingua". Este notebook responde a tres questoes centrais:

1. **Quem somos?** - Caracterizacao dos colaboradores (idade, genero, departamento, cargo)
2. **Somos felizes?** - Analise de satisfacao e factores que influenciam o bem-estar
3. **Igualdade de genero** - Quao perto estamos da meta de 50% de mulheres em todos os cargos?

Adicionalmente, avaliamos o **impacto da introducao de uma mesa de ping pong** na satisfacao dos colaboradores, comparando dados antes e depois da medida.

---
## PARTE 0 - Setup do Ambiente

Como estamos a usar o Google Colab com kernel R, precisamos primeiro de montar o Google Drive (celula Python) e depois carregar os pacotes e ler os CSVs em R.

In [ ]:
# Instalar pacotes que nao vem no Colab por defeito
install.packages(c("corrplot", "gridExtra", "scales"), quiet = TRUE)
# Carregar pacotes
library(tidyverse)
library(corrplot)
library(gridExtra)
library(scales)

In [ ]:
# ============================================================
# IBM Carbon Design System — Paleta de Cores para Data Viz
# https://carbondesignsystem.com/data-visualization/color-palettes/
#
# Todas as cores dos graficos sao definidas aqui como constantes.
# Para ajustar o esquema visual, modifica apenas esta celula.
# ============================================================

# Cor primaria IBM — barras e histogramas simples
ibm_blue     <- "#0f62fe"   # blue-60

# Paleta categorical (sequencia oficial Carbon, light theme)
# Usar sempre nesta ordem para maximizar contraste entre categorias
ibm_cat <- c(
  "#6929c4",   # 1 — purple-70
  "#1192e8",   # 2 — cyan-50
  "#005d5d",   # 3 — teal-70
  "#9f1853",   # 4 — magenta-70
  "#fa4d56",   # 5 — red-50
  "#198038"    # 6 — green-60
)

# Cores semanticas
ibm_positive <- "#198038"   # green-60  — retencao, resultado bom
ibm_negative <- "#fa4d56"   # red-50    — attrition, alerta, outlier
ibm_warning  <- "#f1c21b"   # yellow-30 — atencao

# Departamentos (named vector — ordem explicita)
ibm_dept <- c(
  "Human Resources"        = "#6929c4",  # purple-70
  "Research & Development" = "#1192e8",  # cyan-50
  "Sales"                  = "#005d5d"   # teal-70
)

# Genero (named vector)
ibm_gender <- c(
  "Female" = "#9f1853",   # magenta-70
  "Male"   = "#1192e8"    # cyan-50
)

# Escala de satisfacao ordinal (1 = Low -> 4 = Very High)
ibm_satisfacao <- c(
  "1" = "#fa4d56",   # red-50    — Low
  "2" = "#f1c21b",   # yellow-30 — Medium
  "3" = "#009d9a",   # teal-50   — High
  "4" = "#198038"    # green-60  — Very High
)

# Antes / Depois (Ping Pong Survey)
ibm_periodo <- c(
  "Antes"  = "#0f62fe",   # blue-60
  "Depois" = "#08bdba"    # teal-40
)

# JobLevel — escala sequencial de azuis IBM (1=junior, 5=senior)
ibm_joblevel <- c(
  "1" = "#a6c8ff",   # blue-30
  "2" = "#4589ff",   # blue-50
  "3" = "#0f62fe",   # blue-60
  "4" = "#0043ce",   # blue-70
  "5" = "#002d9c"    # blue-80
)

# Gradiente divergente para matriz de correlacao
ibm_corr_pal <- colorRampPalette(c(ibm_negative, "white", ibm_blue))(200)

cat("Paleta IBM Carbon carregada.\n")


In [ ]:
# ============================================================
# OPCAO 1: Upload manual para o Colab
#   - Faz upload dos 2 CSVs pelo painel de ficheiros do Colab
#     (icone de pasta > Upload to session storage)
#   - Os ficheiros ficam em /content/ automaticamente
# ============================================================

# data_path   <- "/content/"
# df_original <- read_csv(file.path(data_path, "csv_original_projeto_1.csv"), show_col_types = FALSE)
# df_survey   <- read_csv(file.path(data_path, "PingPongSurvey.csv"),           show_col_types = FALSE)

# ============================================================
# OPCAO 2: Carregar directamente do GitHub (sem upload manual)
#   - Substitui os URLs pelos links raw do repositorio
#   - Comenta as linhas da Opcao 1 acima e descomenta as abaixo
# ============================================================

url_original <- "https://raw.githubusercontent.com/diogo-costa-silva/ibm-hr-analytics/refs/heads/main/data/csv_original_projeto_1.csv"
url_survey   <- "https://raw.githubusercontent.com/diogo-costa-silva/ibm-hr-analytics/refs/heads/main/data/PingPongSurvey.csv"
#
df_original <- read_csv(url_original, show_col_types = FALSE)
df_survey   <- read_csv(url_survey,   show_col_types = FALSE)

# ------------------------------------------------------------
cat("CSV Original:", nrow(df_original), "linhas x", ncol(df_original), "colunas\n")
cat("Ping Pong Survey:", nrow(df_survey), "linhas x", ncol(df_survey), "colunas\n")


In [ ]:
glimpse(df_original)

In [ ]:
glimpse(df_survey)

---
## PARTE 1 - Limpeza Inicial

O formador referiu duas dicas importantes:
- **Dica 3:** *"Tirar todas as variaveis com desvio padrao zero"*
- **Dica 11:** *"Remover coluna count"*

**Porque?** Colunas constantes nao trazem informacao analitica - se todos os valores sao iguais, a variavel nao ajuda a distinguir nada.

In [ ]:
# Identificar colunas numericas com desvio padrao = 0
colunas_numericas <- df_original %>%
  select(where(is.numeric)) %>%
  names()

colunas_constantes <- colunas_numericas[
  sapply(df_original[colunas_numericas], sd, na.rm = TRUE) == 0
]

cat("Colunas com desvio padrao zero (constantes):\n")
for (col in colunas_constantes) {
  cat("  -", col, "=", unique(df_original[[col]]), "\n")
}
cat("  - Over18 = Y (texto, nao capturado pelo sd)\n")

In [ ]:
# Remover colunas constantes
df <- df_original %>%
  select(-EmployeeCount, -StandardHours, -Over18)

cat("Antes:", ncol(df_original), "colunas\n")
cat("Depois:", ncol(df), "colunas\n")
cat("Removidas:", ncol(df_original) - ncol(df), "colunas\n")

---
## PARTE 2 - Estatisticas Descritivas

Nesta seccao analisamos: medias, medianas, desvio padrao, minimos, maximos e quartis de todas as variaveis numericas, assim como as frequencias de todas as variaveis categoricas.

### 2.1 - Resumo das Variaveis Numericas

In [ ]:
# Resumo estatistico de todas as variaveis numericas
resumo_numerico <- df %>%
  select(where(is.numeric), -EmployeeNumber) %>%
  pivot_longer(everything(), names_to = "variavel", values_to = "valor") %>%
  group_by(variavel) %>%
  summarise(
    n       = n(),
    media   = round(mean(valor, na.rm = TRUE), 2),
    mediana = median(valor, na.rm = TRUE),
    desvio  = round(sd(valor, na.rm = TRUE), 2),
    minimo  = min(valor, na.rm = TRUE),
    maximo  = max(valor, na.rm = TRUE),
    q1      = quantile(valor, 0.25, na.rm = TRUE),
    q3      = quantile(valor, 0.75, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(variavel)

print(resumo_numerico, n = 30)

### 2.2 - Resumo das Variaveis Categoricas

In [ ]:
# Frequencias de cada variavel categorica
variaveis_cat <- df %>%
  select(where(is.character)) %>%
  names()

for (col in variaveis_cat) {
  cat("\n", col, ":\n")
  freq <- table(df[[col]])
  prop <- round(prop.table(freq) * 100, 1)
  for (i in seq_along(freq)) {
    cat("  ", names(freq)[i], ":", freq[i], "(", prop[i], "%)\n")
  }
}

### 2.3 - Dados Ausentes (Missing Values)

In [ ]:
# Verificacao de NAs
total_na <- sum(is.na(df))
cat("Total de valores NA no dataset:", total_na, "\n")

if (total_na > 0) {
  na_por_coluna <- colSums(is.na(df))
  print(na_por_coluna[na_por_coluna > 0])
} else {
  cat("Sem dados ausentes - dataset completo!\n")
}

### 2.4 - Nota sobre PerformanceRating

**Observacao importante:** A escala de PerformanceRating vai de 1 a 4, mas **nenhum colaborador tem rating 1 (Low) ou 2 (Good)**. Apenas existem valores 3 (Excellent) e 4 (Outstanding).

Possiveis explicacoes:
- **Inflacao de ratings** - a empresa e generosa nas avaliacoes
- **Survivorship bias** - quem tinha ratings baixos ja saiu da empresa
- **Criterios pouco diferenciadores** - a escala nao discrimina o suficiente

In [ ]:
cat("Distribuicao do PerformanceRating:\n")
print(table(df$PerformanceRating))
cat("\nApenas valores 3 (Excellent) e 4 (Outstanding) - sem ratings 1 ou 2!\n")

---
## PARTE 3 - Distribuicoes (Histogramas + Boxplots)

O formador pediu especificamente:
- *"Box plots - level, cargo, income"*
- *"O salario e o level estao relacionados"*

### 3.1 - Distribuicao de Idade e Salario Mensal

In [ ]:
# Histogramas: Idade e Salario Mensal
p_age_hist <- ggplot(df, aes(x = Age)) +
  geom_histogram(binwidth = 2, fill = ibm_blue, color = "white", alpha = 0.8) +
  labs(title = "Distribuicao de Idade", x = "Idade", y = "Contagem") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

p_income_hist <- ggplot(df, aes(x = MonthlyIncome)) +
  geom_histogram(binwidth = 1000, fill = ibm_blue, color = "white", alpha = 0.8) +
  labs(title = "Distribuicao do Salario Mensal",
       x = "Salario Mensal", y = "Contagem") +
  scale_x_continuous(labels = comma) +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

grid.arrange(p_age_hist, p_income_hist, ncol = 2)

### 3.2 - Salario Mensal por Job Level

Esta e a relacao que o formador referiu: **salario e level estao diretamente relacionados**. O boxplot confirma que cada nivel hierarquico tem uma faixa salarial bem distinta.

In [ ]:
# Boxplot: MonthlyIncome por JobLevel
ggplot(df, aes(x = factor(JobLevel), y = MonthlyIncome, fill = factor(JobLevel))) +
  geom_boxplot(alpha = 0.7, outlier.color = ibm_negative, outlier.size = 2) +
  scale_fill_manual(values = ibm_joblevel) +
  labs(title = "Salario Mensal por Job Level",
       subtitle = "Ha uma relacao clara entre level e salario",
       x = "Job Level", y = "Salario Mensal") +
  scale_y_continuous(labels = comma) +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "none")

### 3.3 - Salario Mensal por Cargo

In [ ]:
# Boxplot: MonthlyIncome por JobRole
ggplot(df, aes(x = reorder(JobRole, MonthlyIncome, FUN = median), y = MonthlyIncome)) +
  geom_boxplot(fill = ibm_blue, alpha = 0.7, outlier.color = ibm_negative) +
  labs(title = "Salario Mensal por Cargo", x = "", y = "Salario Mensal") +
  scale_y_continuous(labels = comma) +
  coord_flip() +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

### 3.4 - Idade por Departamento

In [ ]:
# Boxplot: Idade por Department
ggplot(df, aes(x = Department, y = Age, fill = Department)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_manual(values = ibm_dept) +
  labs(title = "Distribuicao de Idade por Departamento", x = "", y = "Idade") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "none")

### 3.5 - Anos na Empresa

In [ ]:
# Histograma: YearsAtCompany
ggplot(df, aes(x = YearsAtCompany)) +
  geom_histogram(binwidth = 1, fill = ibm_blue, color = "white", alpha = 0.8) +
  labs(title = "Distribuicao de Anos na Empresa",
       x = "Anos na Empresa", y = "Contagem") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

---
## PARTE 4 - Categoricas (Bar Plots)

Aqui respondemos as questoes do cliente:
- Quantos colaboradores por departamento?
- Qual a distribuicao de genero? Quao longe estamos da **meta de 50%**?
- Qual a taxa de attrition (turnover)?

### 4.1 - Departamento e Genero

In [ ]:
# Distribuicao por Departamento
p_dept <- ggplot(df, aes(x = reorder(Department, Department, function(x) -length(x)))) +
  geom_bar(fill = ibm_blue, alpha = 0.8) +
  geom_text(stat = "count", aes(label = after_stat(count)), vjust = -0.5, size = 3.5) +
  labs(title = "Colaboradores por Departamento", x = "", y = "Contagem") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

# Distribuicao por Genero (com linha da meta 50%)
p_gender <- ggplot(df, aes(x = Gender, fill = Gender)) +
  geom_bar(alpha = 0.8) +
  geom_text(stat = "count", aes(label = paste0(after_stat(count), "\n(",
            round(after_stat(count)/nrow(df)*100, 1), "%)")),
            vjust = -0.3, size = 3.5) +
  scale_fill_manual(values = ibm_gender) +
  geom_hline(yintercept = nrow(df) / 2, linetype = "dashed", color = ibm_negative,
             linewidth = 0.8) +
  annotate("text", x = 2.4, y = nrow(df)/2 + 30, label = "Meta 50%",
           color = ibm_negative, size = 3.5, fontface = "italic") +
  labs(title = "Distribuicao por Genero",
       subtitle = "Meta: 50% mulheres em todos os cargos",
       x = "", y = "Contagem") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "none")

grid.arrange(p_dept, p_gender, ncol = 2)

### 4.2 - Genero por Departamento

A linha vermelha tracejada marca a **meta de 50% de mulheres** em cada departamento.

In [ ]:
# Proporcao de Genero por Departamento
df %>%
  count(Department, Gender) %>%
  group_by(Department) %>%
  mutate(pct = n / sum(n) * 100) %>%
  ggplot(aes(x = Department, y = pct, fill = Gender)) +
  geom_col(alpha = 0.8) +
  geom_text(aes(label = paste0(round(pct, 1), "%")),
            position = position_stack(vjust = 0.5), size = 3.5) +
  geom_hline(yintercept = 50, linetype = "dashed", color = ibm_negative, linewidth = 0.8) +
  scale_fill_manual(values = ibm_gender) +
  labs(title = "Proporcao de Genero por Departamento",
       subtitle = "Linha vermelha = meta de 50%",
       x = "", y = "Percentagem (%)") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

### 4.3 - Taxa de Attrition (Turnover)

In [ ]:
# Attrition Rate
df %>%
  count(Attrition) %>%
  mutate(pct = n / sum(n) * 100) %>%
  ggplot(aes(x = Attrition, y = n, fill = Attrition)) +
  geom_col(alpha = 0.8) +
  geom_text(aes(label = paste0(n, " (", round(pct, 1), "%)")),
            vjust = -0.3, size = 4) +
  scale_fill_manual(values = c("No" = ibm_positive, "Yes" = ibm_negative)) +
  labs(title = "Taxa de Attrition (Turnover)",
       subtitle = paste0("Attrition Rate: ",
                         round(sum(df$Attrition == "Yes") / nrow(df) * 100, 1), "%"),
       x = "", y = "Contagem") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "none")

### 4.4 - Colaboradores por Cargo

In [ ]:
# Distribuicao por JobRole
ggplot(df, aes(x = reorder(JobRole, JobRole, function(x) length(x)))) +
  geom_bar(fill = ibm_blue, alpha = 0.8) +
  geom_text(stat = "count", aes(label = after_stat(count)), hjust = -0.2, size = 3.5) +
  labs(title = "Colaboradores por Cargo", x = "", y = "Contagem") +
  coord_flip() +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

### 4.5 - Distribuicao das Variaveis de Satisfacao (Antes do Ping Pong)

In [ ]:
# Todas as variaveis de satisfacao num grafico facetado
satisfacao_vars <- c("EnvironmentSatisfaction", "JobSatisfaction",
                     "JobInvolvement", "RelationshipSatisfaction", "WorkLifeBalance")

df %>%
  select(all_of(satisfacao_vars)) %>%
  pivot_longer(everything(), names_to = "Metrica", values_to = "Score") %>%
  mutate(
    Metrica = str_replace_all(Metrica, "([a-z])([A-Z])", "\\1 \\2")
  ) %>%
  ggplot(aes(x = factor(Score), fill = factor(Score))) +
  geom_bar(alpha = 0.8) +
  facet_wrap(~ Metrica, scales = "free_y", ncol = 3) +
  scale_fill_manual(values = ibm_satisfacao,
                    labels = c("Low", "Medium", "High", "Very High")) +
  labs(title = "Distribuicao das Variaveis de Satisfacao (Antes do Ping Pong)",
       x = "Score", y = "Contagem", fill = "Nivel") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

---
## PARTE 5 - Correlacoes

Vamos calcular a matriz de correlacao de todas as variaveis numericas e identificar as relacoes mais fortes.

> **Nota do formador (dica 8):** *"Nao precisamos de R para mostrar correlacoes antes e depois da mesa de ping pong"* - mas SIM precisamos do R para a analise exploratoria geral!

### 5.1 - Matriz de Correlacao

In [ ]:
# Matriz de correlacao
df_corr <- df %>%
  select(where(is.numeric), -EmployeeNumber) %>%
  cor(use = "complete.obs")

corrplot(df_corr,
         method = "color",
         type = "upper",
         order = "hclust",
         tl.col = "black",
         tl.cex = 0.7,
         col = ibm_corr_pal,
         title = "Matriz de Correlacao - Variaveis Numericas",
         mar = c(0, 0, 2, 0),
         addCoef.col = "black",
         number.cex = 0.5)

### 5.2 - Top 10 Correlacoes Mais Fortes

In [ ]:
# Extrair os pares mais correlacionados
cor_long <- as.data.frame(as.table(df_corr)) %>%
  filter(Var1 != Var2) %>%
  mutate(abs_corr = abs(Freq)) %>%
  arrange(desc(abs_corr)) %>%
  filter(!duplicated(abs_corr)) %>%
  head(10)

cat("Top 10 Correlacoes Mais Fortes:\n\n")
for (i in 1:nrow(cor_long)) {
  cat(sprintf("  %s <-> %s : r = %.3f\n",
              cor_long$Var1[i], cor_long$Var2[i], cor_long$Freq[i]))
}

---
## PARTE 6 - Analise de Attrition (Turnover)

**Attrition e a metrica-alvo do dataset!** Vamos investigar o que distingue os colaboradores que sairam dos que ficaram.

### 6.1 - Attrition por Departamento

In [ ]:
# Taxa de Attrition por Departamento
df %>%
  count(Department, Attrition) %>%
  group_by(Department) %>%
  mutate(pct = n / sum(n) * 100) %>%
  filter(Attrition == "Yes") %>%
  ggplot(aes(x = reorder(Department, pct), y = pct, fill = Department)) +
  geom_col(alpha = 0.8) +
  geom_text(aes(label = paste0(round(pct, 1), "%")), hjust = -0.2, size = 4) +
  scale_fill_manual(values = ibm_dept) +
  labs(title = "Taxa de Attrition por Departamento",
       x = "", y = "Attrition Rate (%)") +
  coord_flip() +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "none")

### 6.2 - Attrition por OverTime

In [ ]:
# Attrition: OverTime vs Non-OverTime
df %>%
  count(OverTime, Attrition) %>%
  group_by(OverTime) %>%
  mutate(pct = n / sum(n) * 100) %>%
  filter(Attrition == "Yes") %>%
  ggplot(aes(x = OverTime, y = pct, fill = OverTime)) +
  geom_col(alpha = 0.8, width = 0.5) +
  geom_text(aes(label = paste0(round(pct, 1), "%")), vjust = -0.3, size = 4) +
  scale_fill_manual(values = c("No" = ibm_positive, "Yes" = ibm_negative)) +
  labs(title = "Attrition Rate: OverTime vs Non-OverTime",
       x = "Trabalha Horas Extra?", y = "Attrition Rate (%)") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "none")

### 6.3 - Comparacao de Medias: Quem saiu vs Quem ficou

In [ ]:
# Tabela comparativa: saiu vs ficou
comparacao <- df %>%
  group_by(Attrition) %>%
  summarise(
    Media_Idade              = round(mean(Age), 1),
    Media_Salario            = round(mean(MonthlyIncome), 0),
    Media_AnosEmpresa        = round(mean(YearsAtCompany), 1),
    Media_DistanciaCasa      = round(mean(DistanceFromHome), 1),
    Media_AnosSemPromocao    = round(mean(YearsSinceLastPromotion), 1),
    Media_SatisfacaoAmbiente = round(mean(EnvironmentSatisfaction), 2),
    Media_SatisfacaoTrabalho = round(mean(JobSatisfaction), 2),
    Media_WorkLifeBalance    = round(mean(WorkLifeBalance), 2),
    Pct_OverTime             = round(mean(OverTime == "Yes") * 100, 1),
    .groups = "drop"
  )

cat("Comparacao: Quem saiu vs Quem ficou\n\n")
print(as.data.frame(comparacao))

### 6.4 - Attrition por Job Level

In [ ]:
# Attrition Rate por Job Level
df %>%
  count(JobLevel, Attrition) %>%
  group_by(JobLevel) %>%
  mutate(pct = n / sum(n) * 100) %>%
  filter(Attrition == "Yes") %>%
  ggplot(aes(x = factor(JobLevel), y = pct)) +
  geom_col(fill = ibm_negative, alpha = 0.8, width = 0.5) +
  geom_text(aes(label = paste0(round(pct, 1), "%")), vjust = -0.3, size = 4) +
  labs(title = "Attrition Rate por Job Level",
       subtitle = "Niveis mais baixos tem maior turnover",
       x = "Job Level", y = "Attrition Rate (%)") +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

---
## PARTE 7 - Comparacao Antes vs Depois (Ping Pong Survey)

O PingPong Survey tem **1233 respostas** - exactamente os colaboradores activos (Attrition = "No"). Quem saiu da empresa (237 pessoas) nao respondeu.

**Ponto analitico importante (dica 10 do formador):** *"Perdi o contingente que gostava de trabalhar?"*

Se quem saiu tinha satisfacao alta, a media pode ter caido nao por causa do ping pong, mas porque perdemos as pessoas mais satisfeitas. Isto e um possivel **confounding factor**.

### 7.1 - Preparar dados e calcular medias

In [ ]:
# Preparar dados: Antes (apenas colaboradores ativos) e Depois (survey)
metricas <- c("EnvironmentSatisfaction", "JobInvolvement", "JobSatisfaction",
              "RelationshipSatisfaction", "WorkLifeBalance")

df_antes  <- df %>% filter(Attrition == "No") %>% select(EmployeeNumber, all_of(metricas))
df_depois <- df_survey %>% select(EmployeeNumber, all_of(metricas))

cat("Antes (ativos):", nrow(df_antes), "colaboradores\n")
cat("Depois (survey):", nrow(df_depois), "colaboradores\n")

# Calcular medias antes e depois
medias_antes <- df_antes %>%
  select(-EmployeeNumber) %>%
  summarise(across(everything(), mean)) %>%
  pivot_longer(everything(), names_to = "Metrica", values_to = "Antes")

medias_depois <- df_depois %>%
  select(-EmployeeNumber) %>%
  summarise(across(everything(), mean)) %>%
  pivot_longer(everything(), names_to = "Metrica", values_to = "Depois")

comparacao_pp <- medias_antes %>%
  inner_join(medias_depois, by = "Metrica") %>%
  mutate(
    Diferenca    = round(Depois - Antes, 3),
    Variacao_pct = round((Depois - Antes) / Antes * 100, 2)
  )

cat("\nComparacao Antes vs Depois do Ping Pong:\n\n")
print(as.data.frame(comparacao_pp))

# Dica 9 do formador: "Caiu a moral media?"
cat("\nMoral media (media de todas as metricas):\n")
cat("  Antes:", round(mean(comparacao_pp$Antes), 3), "\n")
cat("  Depois:", round(mean(comparacao_pp$Depois), 3), "\n")

### 7.2 - Grafico Comparativo

In [ ]:
# Grafico: Antes vs Depois
comparacao_pp %>%
  select(Metrica, Antes, Depois) %>%
  pivot_longer(cols = c(Antes, Depois), names_to = "Periodo", values_to = "Media") %>%
  mutate(
    Metrica = str_replace_all(Metrica, "([a-z])([A-Z])", "\\1 \\2"),
    Periodo = factor(Periodo, levels = c("Antes", "Depois"))
  ) %>%
  ggplot(aes(x = Metrica, y = Media, fill = Periodo)) +
  geom_col(position = position_dodge(width = 0.7), width = 0.6, alpha = 0.8) +
  geom_text(aes(label = round(Media, 2)),
            position = position_dodge(width = 0.7), vjust = -0.3, size = 3.5) +
  scale_fill_manual(values = ibm_periodo) +
  labs(title = "Satisfacao: Antes vs Depois do Ping Pong",
       subtitle = "Comparacao das medias das 5 variaveis de satisfacao",
       x = "", y = "Media (1-4)", fill = "") +
  ylim(0, 4) +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14),
        axis.text.x = element_text(angle = 15, hjust = 1))

### 7.3 - Analise Individual: Quem melhorou, manteve ou piorou?

In [ ]:
# Para cada metrica: % que melhorou, manteve ou piorou
df_individual <- df_antes %>%
  inner_join(df_depois, by = "EmployeeNumber", suffix = c("_antes", "_depois"))

cat("Analise Individual:\n")

for (m in metricas) {
  antes_col  <- paste0(m, "_antes")
  depois_col <- paste0(m, "_depois")

  diff <- df_individual[[depois_col]] - df_individual[[antes_col]]

  melhorou <- sum(diff > 0)
  manteve  <- sum(diff == 0)
  piorou   <- sum(diff < 0)
  total    <- length(diff)

  cat(sprintf("\n%s:\n  Melhorou: %d (%.1f%%)  |  Manteve: %d (%.1f%%)  |  Piorou: %d (%.1f%%)\n",
              m, melhorou, melhorou/total*100, manteve, manteve/total*100, piorou, piorou/total*100))
}

### 7.4 - Contexto Critico: Satisfacao de quem SAIU

Para responder a dica 10 do formador: **perdemos as pessoas mais felizes ou mais infelizes?**

In [ ]:
# Satisfacao media: quem saiu vs quem ficou
df_saiu  <- df %>% filter(Attrition == "Yes") %>% select(all_of(metricas))
df_ficou <- df %>% filter(Attrition == "No")  %>% select(all_of(metricas))

cat("Satisfacao media: Quem saiu vs Quem ficou\n")
cat("(Perdemos as pessoas mais felizes ou mais infelizes?)\n\n")

for (m in metricas) {
  media_saiu  <- round(mean(df_saiu[[m]]), 3)
  media_ficou <- round(mean(df_ficou[[m]]), 3)
  cat(sprintf("  %s:\n    Saiu: %.3f  |  Ficou: %.3f  |  Diff: %+.3f\n",
              m, media_saiu, media_ficou, media_saiu - media_ficou))
}

---
## Analise Exploratoria Concluida!

### Principais Conclusoes

1. **Genero:** 40% feminino vs 60% masculino - longe da meta de 50%
2. **Attrition:** 16.1% - acima do benchmark saudavel (~10-12%)
3. **OverTime:** Quem faz horas extra tem attrition significativamente mais alta
4. **PerformanceRating:** Apenas ratings 3 e 4 - possivel inflacao de avaliacoes
5. **Salario <-> Level:** Relacao direta confirmada; niveis baixos com salarios e attrition mais altos
6. **Ping Pong:** Comparacao antes/depois mostra variacoes nas medias de satisfacao

### Proximos Passos

1. Importar CSVs no SQL Server (VM)
2. Executar o `Projeto1.sql` - criacao do star schema (dimensoes + factos)
3. Conectar Power BI ao SQL Server
4. Criar relatorios detalhados e slides executivos para o C-Level